# Education Outcome Prediction Pipeline

This notebook builds a **resident-month machine learning pipeline** to predict whether a resident is likely to meet an education milestone in the **next month**.

This notebook is organized to tell the full pipeline story expected in the IS 455 project:
1. Problem Framing  
2. Data Acquisition, Preparation & Exploration  
3. Modeling & Feature Selection  
4. Evaluation & Interpretation  
5. Causal and Relationship Analysis  
6. Deployment Notes

The design also follows the broader textbook workflow:
- **data preparation pipeline thinking** from the preparation chapters
- **predictive classification modeling** for a binary target
- **cross-validation and fair model comparison**
- **deployment through saved outputs that the website can read**

## Target definition

`milestone_met_next_month = 1` when the next-month resident education record satisfies both:
- `education_progress >= 70`
- `attendance_rate >= 0.80`

Otherwise, the target is `0`.

## Practical use in the web app

This pipeline is most naturally used in:
- **Caseload Inventory**
- **Reports & Analytics**
- **Admin Dashboard**

The model output can help staff identify which residents may need additional academic support before progress stalls.

## 1. Problem Framing

### Business problem
The organization needs a way to identify residents who may be at risk of missing near-term education milestones. Education progress is one of the clearest outcome signals in the case-management process, and early intervention matters.

### Who cares about this?
This pipeline is useful for:
- staff managing resident progress
- administrators monitoring outcome patterns across safehouses
- leadership reviewing education-related program results

### Why this matters
A lagging education outcome may reflect attendance issues, health challenges, unstable circumstances, or broader case-management strain. Predicting the risk early gives staff a chance to respond sooner.

### Predictive vs. explanatory choice
This notebook is **primarily predictive**. The goal is to estimate which resident-month records are more likely to miss the next-month milestone.  
A separate relationship-analysis section is still included so the notebook can discuss what the model structure reveals, but the main success standard is **out-of-sample performance**, not causal proof.

### Textbook alignment
This notebook is designed to align with:
- **data preparation pipeline structure**
- **dummy handling / encoding through preprocessing pipelines**
- **classification modeling**
- **cross-validation and model comparison**
- **deployment-ready outputs**

## 2. Data Acquisition, Preparation & Exploration

This section loads the relevant INTEX tables, standardizes naming, and builds the resident-month panel used for modeling.

### Data sources used
This notebook looks for and uses the case-management tables that support education monitoring, including:
- resident master data
- education records
- incident reports
- health / wellbeing records
- process recordings
- intervention plans

### Preparation strategy
The preparation flow follows the project pattern:
- discover and load files robustly
- standardize names and key columns
- build a resident-month base table
- aggregate supporting signals from other tables
- create a future outcome target without leakage

### Notebook setup and reusable helper functions

This block sets up:
- imports
- path resolution
- file loading helpers
- name normalization helpers
- month handling helpers

The goal is to make the notebook easier to rerun top-to-bottom and more robust to small schema or file-location differences.

In [1]:

# Core imports and notebook configuration.
import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# Keep output paths stable across reruns.
PROJECT_ROOT = Path.cwd()
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "generated_outputs"
OUTPUT_DIR = Path(os.environ.get("AZUREML_OUTPUT_DIR", str(LOCAL_OUTPUT_DIR)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility setting for all model steps.
SEED = 42

# Make notebook output easier to read.
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines
Output directory: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs


In [2]:

# Helper functions for robust file discovery and standardization.

def normalize_name(value: str) -> str:
    """Return a simplified lowercase name for flexible file and column matching."""
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def candidate_roots() -> list[Path]:
    """Search common local locations for the CSV tables without hard-coding one folder."""
    roots = [
        PROJECT_ROOT.parent / "data",
        PROJECT_ROOT / "data",
        Path("../data"),
        Path("./data"),
        Path("/mnt/batch/tasks/shared/LS_root/mounts/clusters/notebookdev/code/data"),
        Path("/lighthouse/lighthouse_csv_v7/lighthouse_csv_v7"),
        Path("/lighthouse/lighthouse_csv_v7"),
        Path("/lighthouse"),
        PROJECT_ROOT,
        PROJECT_ROOT / "Data",
        PROJECT_ROOT / "csv",
        PROJECT_ROOT / "CSV",
        PROJECT_ROOT / "files",
        PROJECT_ROOT / "dataset",
        PROJECT_ROOT / "datasets",
    ]

    unique_roots = []
    seen = set()
    for root in roots:
        if root.exists():
            resolved = root.resolve()
            if resolved not in seen:
                unique_roots.append(root)
                seen.add(resolved)
    return unique_roots


def discover_csv_files(search_roots: list[Path]) -> dict[str, Path]:
    """Recursively discover CSV files and index them by normalized stem and filename."""
    found = {}
    for root in search_roots:
        for path in root.rglob("*.csv"):
            stem_key = normalize_name(path.stem)
            file_key = normalize_name(path.name)
            found.setdefault(stem_key, path)
            found.setdefault(file_key, path)
    return found


def resolve_table_path(alias_options: list[str], file_index: dict[str, Path]) -> Path | None:
    """Return the first matching path from a list of possible table names."""
    for alias in alias_options:
        key = normalize_name(alias)
        if key in file_index:
            return file_index[key]
    return None


def safe_read_csv(path: Path) -> pd.DataFrame:
    """Read a CSV with a small encoding fallback list."""
    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            last_error = exc
    raise last_error


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names and trim string values."""
    df = df.copy()
    df.columns = [normalize_name(col) for col in df.columns]
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan, "none": np.nan})
    return df


def first_existing(df: pd.DataFrame, options: list[str]) -> str | None:
    """Return the first column that exists from a list of aliases."""
    normalized_map = {normalize_name(col): col for col in df.columns}
    for option in options:
        key = normalize_name(option)
        if key in normalized_map:
            return normalized_map[key]
    return None


def ensure_datetime(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    """Safely convert a column to datetime if it exists."""
    df = df.copy()
    if column_name in df.columns:
        df[column_name] = pd.to_datetime(df[column_name], errors="coerce")
    return df


def month_floor(series: pd.Series) -> pd.Series:
    """Convert a datetime series to first day of month timestamps."""
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()


SEARCH_ROOTS = candidate_roots()
FILE_INDEX = discover_csv_files(SEARCH_ROOTS)

print("Search roots checked:")
for root in SEARCH_ROOTS:
    print(" -", root)

print(f"Discovered CSV entries: {len(FILE_INDEX)}")

Search roots checked:
 - c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines
Discovered CSV entries: 54


### File discovery and table loading

This block defines the likely table aliases and loads whichever matching INTEX tables are available.  
That makes the notebook more tolerant of small file-name differences across team folders.

In [3]:

# Define likely INTEX table aliases and load the available tables.
# Update or extend these aliases if your team uses slightly different file names.

TABLE_ALIASES = {
    "residents": ["residents", "resident", "resident_master"],
    "education_records": ["education_records", "education_record", "education"],
    "incident_reports": ["incident_reports", "incident_report", "incidents", "incident"],
    "health_records": ["health_records", "health_record", "health", "wellbeing_records", "wellbeing"],
    "process_recordings": ["process_recordings", "process_recording", "counseling_sessions", "process_sessions"],
    "intervention_plans": ["intervention_plans", "intervention_plan", "plans"],
    "safehouses": ["safehouses", "safehouse", "homes"],
}

tables = {}
table_paths = {}

for logical_name, aliases in TABLE_ALIASES.items():
    path = resolve_table_path(aliases, FILE_INDEX)
    table_paths[logical_name] = path
    if path is not None:
        df = safe_read_csv(path)
        df = standardize_columns(df)
        tables[logical_name] = df
        print(f"Loaded {logical_name:<20} -> {path.name:<35} shape={df.shape}")
    else:
        print(f"Missing {logical_name:<20} -> no matching CSV found")

if "education_records" not in tables:
    raise FileNotFoundError(
        "No education_records table was found. This notebook requires an education table to build the target."
    )

Loaded residents            -> residents.csv                       shape=(60, 49)
Loaded education_records    -> education_records.csv               shape=(534, 10)
Loaded incident_reports     -> incident_reports.csv                shape=(100, 12)
Missing health_records       -> no matching CSV found
Loaded process_recordings   -> process_recordings.csv              shape=(2819, 15)
Loaded intervention_plans   -> intervention_plans.csv              shape=(180, 11)
Loaded safehouses           -> safehouses.csv                      shape=(9, 13)


### Initial table review

This section gives a quick structural review of the loaded tables so the pipeline has an explicit checkpoint before transformation begins.

In [4]:

# Review the loaded tables before transformation.
table_summary = []

for name, df in tables.items():
    table_summary.append(
        {
            "table_name": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "sample_columns": ", ".join(df.columns[:10]),
        }
    )

table_summary_df = pd.DataFrame(table_summary).sort_values("table_name").reset_index(drop=True)
display(table_summary_df)

# Show a preview of the education table because it is the core source for this pipeline.
display(tables["education_records"].head())

,table_name,rows,columns,sample_columns
0,education_records,534,10,"education_record_id, resident_id, record_date,..."
1,incident_reports,100,12,"incident_id, resident_id, safehouse_id, incide..."
2,intervention_plans,180,11,"plan_id, resident_id, plan_category, plan_desc..."
3,process_recordings,2819,15,"recording_id, resident_id, session_date, socia..."
4,residents,60,49,"resident_id, case_control_no, internal_code, s..."
5,safehouses,9,13,"safehouse_id, safehouse_code, name, region, ci..."


,education_record_id,resident_id,record_date,education_level,school_name,enrollment_status,attendance_rate,progress_percent,completion_status,notes
0,1,1,2023-10-01,Vocational,School 8,Enrolled,0.966,37.7,NotStarted,Progress: NotStarted
1,2,1,2023-11-01,Secondary,School 20,Enrolled,0.693,33.0,InProgress,Progress: InProgress
2,3,1,2023-12-01,Vocational,School 18,Enrolled,0.744,54.0,InProgress,Progress: InProgress
3,4,1,2024-01-01,Primary,School 2,Enrolled,0.681,51.2,InProgress,Progress: InProgress
4,5,1,2024-02-01,Vocational,School 3,Enrolled,0.721,44.2,InProgress,Progress: InProgress


### Column standardization

This block standardizes key field names such as:
- `resident_id`
- `safehouse_id`
- `education_date`
- `attendance_rate`

This is an important preparation step because later modeling cells should depend on stable names rather than raw file-specific labels.

In [5]:

# Standardize key columns across tables.
# This cell creates stable names like resident_id, safehouse_id, and record_month even if the source names vary.

def rename_first_match(df: pd.DataFrame, alias_map: dict[str, list[str]]) -> pd.DataFrame:
    """Rename the first found alias in each alias list to the target column name."""
    df = df.copy()
    current_names = list(df.columns)
    rename_dict = {}
    for target_name, options in alias_map.items():
        match = first_existing(df, options)
        if match is not None and match != target_name:
            rename_dict[match] = target_name
    if rename_dict:
        df = df.rename(columns=rename_dict)
    return df


COMMON_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "record_date": ["record_date", "date", "session_date", "incident_date", "health_date", "plan_date", "education_date"],
    "created_date": ["created_date", "created_at", "create_date"],
}

EDUCATION_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "education_date": ["education_date", "record_date", "date", "progress_date", "report_date"],
    "education_progress": ["education_progress", "progress_score", "progress", "education_score", "academic_progress"],
    "attendance_rate": ["attendance_rate", "attendance", "attendance_pct", "attendance_percent", "school_attendance_rate"],
    "grade_level": ["grade_level", "grade", "current_grade"],
}

INCIDENT_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "incident_date": ["incident_date", "record_date", "date"],
    "severity": ["severity", "incident_severity", "severity_level"],
    "follow_up_required": ["follow_up_required", "requires_follow_up", "follow_up"],
}

HEALTH_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "health_date": ["health_date", "record_date", "date"],
    "health_score": ["health_score", "overall_health_score", "wellbeing_score"],
    "sleep_quality_score": ["sleep_quality_score", "sleep_score"],
    "energy_level_score": ["energy_level_score", "energy_score"],
}

PROCESS_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "session_date": ["session_date", "record_date", "date", "process_recording_date"],
    "session_duration_minutes": ["session_duration_minutes", "duration_minutes", "minutes", "session_length_minutes"],
    "session_type": ["session_type", "type", "counseling_type"],
}

PLAN_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "plan_date": ["plan_date", "record_date", "date", "created_date"],
    "plan_status": ["plan_status", "status"],
    "plan_type": ["plan_type", "type"],
}

SAFEHOUSE_ALIAS_MAP = {
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "name": ["name", "safehouse_name", "home_name"],
    "region": ["region", "area"],
    "occupancy_rate": ["occupancy_rate", "occupancy", "fill_rate"],
    "open_date": ["open_date", "start_date", "opened_date"],
}

for table_name, df in list(tables.items()):
    df = rename_first_match(df, COMMON_ALIAS_MAP)
    if table_name == "education_records":
        df = rename_first_match(df, EDUCATION_ALIAS_MAP)
    elif table_name == "incident_reports":
        df = rename_first_match(df, INCIDENT_ALIAS_MAP)
    elif table_name == "health_records":
        df = rename_first_match(df, HEALTH_ALIAS_MAP)
    elif table_name == "process_recordings":
        df = rename_first_match(df, PROCESS_ALIAS_MAP)
    elif table_name == "intervention_plans":
        df = rename_first_match(df, PLAN_ALIAS_MAP)
    elif table_name == "safehouses":
        df = rename_first_match(df, SAFEHOUSE_ALIAS_MAP)
    tables[table_name] = df

# Convert relevant date columns.
date_candidates = [
    "education_date", "incident_date", "health_date", "session_date", "plan_date", "open_date",
    "record_date", "created_date"
]
for table_name, df in list(tables.items()):
    for date_col in date_candidates:
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    tables[table_name] = df

for name, df in tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


residents
['resident_id', 'case_control_no', 'internal_code', 'safehouse_id', 'case_status', 'sex', 'date_of_birth', 'birth_status', 'place_of_birth', 'religion', 'case_category', 'sub_cat_orphaned', 'sub_cat_trafficked', 'sub_cat_child_labor', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse', 'sub_cat_osaec', 'sub_cat_cicl', 'sub_cat_at_risk', 'sub_cat_street_child', 'sub_cat_child_with_hiv', 'is_pwd', 'pwd_type', 'has_special_needs', 'special_needs_diagnosis', 'family_is_4ps', 'family_solo_parent', 'family_indigenous', 'family_parent_pwd', 'family_informal_settler', 'date_of_admission', 'age_upon_admission', 'present_age', 'length_of_stay', 'referral_source', 'referring_agency_person', 'date_colb_registered', 'date_colb_obtained', 'assigned_social_worker', 'initial_case_assessment', 'date_case_study_prepared', 'reintegration_type', 'reintegration_status', 'initial_risk_level', 'current_risk_level', 'date_enrolled', 'date_closed', 'created_date', 'notes_restricted']

education_record

### Base education resident-month table

This block creates the core resident-month table from the education records.  
It establishes the monthly time grain used throughout the rest of the notebook.

In [6]:

# Build the education resident-month base table.
education = tables["education_records"].copy()

# Create the monthly timestamp used throughout the notebook.
education["metric_month"] = month_floor(education["education_date"])
education = education.dropna(subset=["resident_id", "metric_month"]).copy()

# Rename the raw progress field used in this dataset.
education = education.rename(columns={"progress_percent": "education_progress"})

# Convert likely numeric fields.
for col in ["education_progress", "attendance_rate"]:
    if col in education.columns:
        education[col] = pd.to_numeric(education[col], errors="coerce")

# Attendance is sometimes stored as 0-100 instead of 0-1.
if "attendance_rate" in education.columns:
    attendance_max = education["attendance_rate"].dropna().max()
    if pd.notna(attendance_max) and attendance_max > 1.5:
        education["attendance_rate"] = education["attendance_rate"] / 100.0

print("Columns in education after rename:")
print(education.columns.tolist())

education_month = (
    education
    .groupby(["resident_id", "metric_month"], as_index=False)
    .agg({
        "education_progress": "mean",
        "attendance_rate": "mean"
    })
)

education_counts = (
    education
    .groupby(["resident_id", "metric_month"], as_index=False)
    .size()
    .rename(columns={"size": "education_record_count"})
)

education_month = education_month.merge(
    education_counts,
    on=["resident_id", "metric_month"],
    how="left"
)

print("Columns in education_month after build:")
print(education_month.columns.tolist())

display(education_month.head())

Columns in education after rename:
['education_record_id', 'resident_id', 'education_date', 'education_level', 'school_name', 'enrollment_status', 'attendance_rate', 'education_progress', 'completion_status', 'notes', 'metric_month']
Columns in education_month after build:
['resident_id', 'metric_month', 'education_progress', 'attendance_rate', 'education_record_count']


,resident_id,metric_month,education_progress,attendance_rate,education_record_count
0,1,2023-10-01,37.7,0.966,1
1,1,2023-11-01,33.0,0.693,1
2,1,2023-12-01,54.0,0.744,1
3,1,2024-01-01,51.2,0.681,1
4,1,2024-02-01,44.2,0.721,1


### Resident master enrichment

This block adds resident and safehouse context when the resident master table is available.
Typical examples include age, admission timing, education level, and safehouse assignment.

In [7]:

# Add resident master data when available.
resident_month = education_month.copy()

if "residents" in tables:
    residents = tables["residents"].copy()

    resident_keep_cols = [
        col for col in [
            "resident_id",
            "safehouse_id",
            "age",
            "current_age",
            "education_level",
            "admission_date",
            "birth_date",
        ] if col in residents.columns
    ]

    if resident_keep_cols:
        residents = residents[resident_keep_cols].drop_duplicates(subset=["resident_id"])
        residents = rename_first_match(
            residents,
            {
                "age": ["age", "current_age"],
                "education_level": ["education_level", "school_level"],
                "admission_date": ["admission_date", "intake_date", "entry_date"],
                "birth_date": ["birth_date", "dob", "date_of_birth"],
            }
        )

        for date_col in ["admission_date", "birth_date"]:
            if date_col in residents.columns:
                residents[date_col] = pd.to_datetime(residents[date_col], errors="coerce")

        resident_month = resident_month.merge(
            residents,
            on="resident_id",
            how="left",
            suffixes=("", "_resident")
        )

        if "safehouse_id" not in resident_month.columns and "safehouse_id_resident" in resident_month.columns:
            resident_month = resident_month.rename(columns={"safehouse_id_resident": "safehouse_id"})
        elif "safehouse_id_resident" in resident_month.columns:
            resident_month["safehouse_id"] = resident_month["safehouse_id"].fillna(resident_month["safehouse_id_resident"])
            resident_month = resident_month.drop(columns=["safehouse_id_resident"])

        if "birth_date" in resident_month.columns and "age" not in resident_month.columns:
            resident_month["age"] = (
                (resident_month["metric_month"] - resident_month["birth_date"]).dt.days / 365.25
            )

        if "admission_date" in resident_month.columns:
            resident_month["months_since_admission"] = (
                (resident_month["metric_month"] - resident_month["admission_date"]).dt.days / 30.44
            )

display(resident_month.head())

,resident_id,metric_month,education_progress,attendance_rate,education_record_count,safehouse_id
0,1,2023-10-01,37.7,0.966,1,4
1,1,2023-11-01,33.0,0.693,1,4
2,1,2023-12-01,54.0,0.744,1,4
3,1,2024-01-01,51.2,0.681,1,4
4,1,2024-02-01,44.2,0.721,1,4


### Monthly aggregation helper

The next set of blocks uses a shared helper pattern to aggregate supporting case-management tables to the resident-month level.
This keeps the feature engineering more consistent and easier to maintain.

In [8]:

# Helper used to build monthly feature aggregates from supporting tables.

def build_monthly_feature_table(
    df: pd.DataFrame,
    date_col: str,
    value_build_fn,
    group_cols: list[str] = ["resident_id", "metric_month"]
) -> pd.DataFrame:
    """Apply a table-specific aggregation function after creating metric_month."""
    work = df.copy()
    if date_col not in work.columns or "resident_id" not in work.columns:
        return pd.DataFrame(columns=group_cols)
    work["metric_month"] = month_floor(work[date_col])
    work = work.dropna(subset=["resident_id", "metric_month"]).copy()
    return value_build_fn(work)

### Incident features

This block rolls incident history up to the resident-month level.
These features help capture whether case instability may be associated with future education performance.

In [9]:

# Aggregate monthly incident features.
incident_month = pd.DataFrame(columns=["resident_id", "metric_month"])

if "incident_reports" in tables:
    incidents = tables["incident_reports"].copy()

    def incident_aggregator(work: pd.DataFrame) -> pd.DataFrame:
        output = (
            work
            .groupby(["resident_id", "metric_month"], as_index=False)
            .size()
            .rename(columns={"size": "incident_count"})
        )

        if "severity" in work.columns:
            severity_map = work["severity"].astype(str).str.lower()
            work["high_severity_flag"] = severity_map.isin(["high", "critical", "severe"]).astype(int)
            work["medium_severity_flag"] = severity_map.isin(["medium", "moderate"]).astype(int)

            severity_agg = (
                work
                .groupby(["resident_id", "metric_month"], as_index=False)[["high_severity_flag", "medium_severity_flag"]]
                .sum()
                .rename(columns={
                    "high_severity_flag": "high_severity_incidents",
                    "medium_severity_flag": "medium_severity_incidents",
                })
            )
            output = output.merge(severity_agg, on=["resident_id", "metric_month"], how="left")

        if "follow_up_required" in work.columns:
            follow_map = work["follow_up_required"].astype(str).str.lower()
            work["follow_up_flag"] = follow_map.isin(["1", "true", "yes", "y"]).astype(int)
            follow_agg = (
                work
                .groupby(["resident_id", "metric_month"], as_index=False)["follow_up_flag"]
                .sum()
                .rename(columns={"follow_up_flag": "follow_up_incidents"})
            )
            output = output.merge(follow_agg, on=["resident_id", "metric_month"], how="left")

        if "safehouse_id" in work.columns:
            safehouse_lookup = (
                work
                .sort_values(["resident_id", "metric_month"])
                .groupby(["resident_id", "metric_month"], as_index=False)
                .tail(1)[["resident_id", "metric_month", "safehouse_id"]]
            )
            output = output.merge(safehouse_lookup, on=["resident_id", "metric_month"], how="left")

        return output

    date_col = "incident_date" if "incident_date" in incidents.columns else "record_date"
    incident_month = build_monthly_feature_table(incidents, date_col, incident_aggregator)

display(incident_month.head())

,resident_id,metric_month,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents,safehouse_id
0,1,2024-02-01,1,0,0,0,4
1,1,2024-06-01,1,0,1,0,4
2,1,2025-04-01,1,0,0,0,4
3,1,2026-02-01,1,1,0,1,4
4,3,2025-02-01,1,0,0,1,1


### Health and wellbeing features

This block aggregates health-related scores to the resident-month level.
These features help represent whether resident wellbeing may be associated with near-term education outcomes.

In [10]:

# Aggregate monthly health features.
health_month = pd.DataFrame(columns=["resident_id", "metric_month"])

if "health_records" in tables:
    health = tables["health_records"].copy()

    def health_aggregator(work: pd.DataFrame) -> pd.DataFrame:
        numeric_candidates = [
            col for col in ["health_score", "sleep_quality_score", "energy_level_score"]
            if col in work.columns
        ]
        for col in numeric_candidates:
            work[col] = pd.to_numeric(work[col], errors="coerce")

        agg_spec = {col: "mean" for col in numeric_candidates}
        output = (
            work
            .groupby(["resident_id", "metric_month"], as_index=False)
            .agg(agg_spec)
        ) if agg_spec else work[["resident_id", "metric_month"]].drop_duplicates()

        count_df = (
            work
            .groupby(["resident_id", "metric_month"], as_index=False)
            .size()
            .rename(columns={"size": "health_record_count"})
        )

        output = output.merge(count_df, on=["resident_id", "metric_month"], how="left")

        if "safehouse_id" in work.columns:
            safehouse_lookup = (
                work
                .sort_values(["resident_id", "metric_month"])
                .groupby(["resident_id", "metric_month"], as_index=False)
                .tail(1)[["resident_id", "metric_month", "safehouse_id"]]
            )
            output = output.merge(safehouse_lookup, on=["resident_id", "metric_month"], how="left")

        return output

    date_col = "health_date" if "health_date" in health.columns else "record_date"
    health_month = build_monthly_feature_table(health, date_col, health_aggregator)

display(health_month.head())

,resident_id,metric_month


### Counseling and process-recording features

This block summarizes process-recording intensity and session characteristics by resident-month.
These features help reflect recent engagement in the healing or support process.

In [11]:

# Aggregate monthly counseling and process recording features.
process_month = pd.DataFrame(columns=["resident_id", "metric_month"])

if "process_recordings" in tables:
    process_df = tables["process_recordings"].copy()

    def process_aggregator(work: pd.DataFrame) -> pd.DataFrame:
        output = (
            work
            .groupby(["resident_id", "metric_month"], as_index=False)
            .size()
            .rename(columns={"size": "process_session_count"})
        )

        if "session_duration_minutes" in work.columns:
            work["session_duration_minutes"] = pd.to_numeric(work["session_duration_minutes"], errors="coerce")
            duration_df = (
                work
                .groupby(["resident_id", "metric_month"], as_index=False)["session_duration_minutes"]
                .mean()
                .rename(columns={"session_duration_minutes": "avg_session_duration"})
            )
            output = output.merge(duration_df, on=["resident_id", "metric_month"], how="left")

        if "session_type" in work.columns:
            session_type_map = work["session_type"].astype(str).str.lower()
            work["group_session_flag"] = session_type_map.str.contains("group", na=False).astype(int)
            group_df = (
                work
                .groupby(["resident_id", "metric_month"], as_index=False)["group_session_flag"]
                .mean()
                .rename(columns={"group_session_flag": "group_session_share"})
            )
            output = output.merge(group_df, on=["resident_id", "metric_month"], how="left")

        if "safehouse_id" in work.columns:
            safehouse_lookup = (
                work
                .sort_values(["resident_id", "metric_month"])
                .groupby(["resident_id", "metric_month"], as_index=False)
                .tail(1)[["resident_id", "metric_month", "safehouse_id"]]
            )
            output = output.merge(safehouse_lookup, on=["resident_id", "metric_month"], how="left")

        return output

    date_col = "session_date" if "session_date" in process_df.columns else "record_date"
    process_month = build_monthly_feature_table(process_df, date_col, process_aggregator)

display(process_month.head())

,resident_id,metric_month,process_session_count,avg_session_duration,group_session_share
0,1,2023-11-01,3,74.000000,0.333333
1,1,2023-12-01,6,75.333333,0.833333
2,1,2024-01-01,3,60.666667,0.333333
3,1,2024-02-01,3,44.333333,0.333333
4,1,2024-03-01,5,71.400000,0.600000


### Intervention-plan features

This block summarizes intervention planning activity by resident-month.
These features can help indicate whether structured support planning is related to future education progress.

In [12]:

# Aggregate monthly intervention plan features.
plan_month = pd.DataFrame(columns=["resident_id", "metric_month"])

if "intervention_plans" in tables:
    plans = tables["intervention_plans"].copy()

    def plan_aggregator(work: pd.DataFrame) -> pd.DataFrame:
        output = (
            work
            .groupby(["resident_id", "metric_month"], as_index=False)
            .size()
            .rename(columns={"size": "plans_created"})
        )

        if "plan_status" in work.columns:
            status_map = work["plan_status"].astype(str).str.lower()
            work["achieved_plan_flag"] = status_map.isin(["achieved", "completed", "done", "closed"]).astype(int)
            achieved_df = (
                work
                .groupby(["resident_id", "metric_month"], as_index=False)["achieved_plan_flag"]
                .sum()
                .rename(columns={"achieved_plan_flag": "achieved_plans"})
            )
            output = output.merge(achieved_df, on=["resident_id", "metric_month"], how="left")

        if "plan_type" in work.columns:
            type_map = work["plan_type"].astype(str).str.lower()
            work["education_plan_flag"] = type_map.str.contains("education", na=False).astype(int)
            education_plan_df = (
                work
                .groupby(["resident_id", "metric_month"], as_index=False)["education_plan_flag"]
                .sum()
                .rename(columns={"education_plan_flag": "education_plans_created"})
            )
            output = output.merge(education_plan_df, on=["resident_id", "metric_month"], how="left")

        if "safehouse_id" in work.columns:
            safehouse_lookup = (
                work
                .sort_values(["resident_id", "metric_month"])
                .groupby(["resident_id", "metric_month"], as_index=False)
                .tail(1)[["resident_id", "metric_month", "safehouse_id"]]
            )
            output = output.merge(safehouse_lookup, on=["resident_id", "metric_month"], how="left")

        return output

    date_col = "plan_date" if "plan_date" in plans.columns else "record_date"
    plan_month = build_monthly_feature_table(plans, date_col, plan_aggregator)

display(plan_month.head())

,resident_id,metric_month,plans_created,achieved_plans
0,1,2023-10-01,3,0
1,2,2023-03-01,3,1
2,3,2024-05-01,3,0
3,4,2024-09-01,3,0
4,5,2024-01-01,3,1


### Merge all monthly features into one modeling panel

This block combines the resident-month base table with the aggregated supporting feature tables.
This creates the main panel that will later receive the prediction target and modeling features.

In [13]:

# Merge all resident-month features into one panel.
resident_month_panel = resident_month.copy()

monthly_feature_tables = [
    incident_month,
    health_month,
    process_month,
    plan_month,
]

for feature_df in monthly_feature_tables:
    if feature_df is None or feature_df.empty:
        continue

    merge_keys = [col for col in ["resident_id", "metric_month"] if col in feature_df.columns]
    safehouse_present = "safehouse_id" in feature_df.columns and "safehouse_id" not in merge_keys

    merge_cols = [col for col in feature_df.columns if col in merge_keys or col != "safehouse_id"]
    feature_subset = feature_df[merge_cols].copy()

    resident_month_panel = resident_month_panel.merge(
        feature_subset,
        on=merge_keys,
        how="left"
    )

    if safehouse_present and "safehouse_id" in feature_df.columns and "safehouse_id" in resident_month_panel.columns:
        safehouse_lookup = feature_df[["resident_id", "metric_month", "safehouse_id"]].drop_duplicates()
        resident_month_panel = resident_month_panel.merge(
            safehouse_lookup,
            on=["resident_id", "metric_month"],
            how="left",
            suffixes=("", "_from_feature")
        )
        if "safehouse_id_from_feature" in resident_month_panel.columns:
            resident_month_panel["safehouse_id"] = resident_month_panel["safehouse_id"].fillna(
                resident_month_panel["safehouse_id_from_feature"]
            )
            resident_month_panel = resident_month_panel.drop(columns=["safehouse_id_from_feature"])

# Add safehouse context if available.
if "safehouses" in tables and "safehouse_id" in resident_month_panel.columns:
    safehouses = tables["safehouses"].copy()
    safehouse_keep_cols = [
        col for col in ["safehouse_id", "name", "region", "occupancy_rate", "open_date"]
        if col in safehouses.columns
    ]
    safehouses = safehouses[safehouse_keep_cols].drop_duplicates(subset=["safehouse_id"])

    if "occupancy_rate" in safehouses.columns:
        safehouses["occupancy_rate"] = pd.to_numeric(safehouses["occupancy_rate"], errors="coerce")
        if safehouses["occupancy_rate"].dropna().max() > 1.5:
            safehouses["occupancy_rate"] = safehouses["occupancy_rate"] / 100.0

    if "open_date" in safehouses.columns:
        safehouses["open_date"] = pd.to_datetime(safehouses["open_date"], errors="coerce")

    resident_month_panel = resident_month_panel.merge(
        safehouses,
        on="safehouse_id",
        how="left"
    )

display(resident_month_panel.head())
print("Panel shape after feature merges:", resident_month_panel.shape)

,resident_id,metric_month,education_progress,attendance_rate,education_record_count,safehouse_id,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents,process_session_count,avg_session_duration,group_session_share,plans_created,achieved_plans,name,region,open_date
0,1,2023-10-01,37.7,0.966,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16
1,1,2023-11-01,33.0,0.693,1,4,NaN,NaN,NaN,NaN,3.0,74.000000,0.333333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16
2,1,2023-12-01,54.0,0.744,1,4,NaN,NaN,NaN,NaN,6.0,75.333333,0.833333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16
3,1,2024-01-01,51.2,0.681,1,4,NaN,NaN,NaN,NaN,3.0,60.666667,0.333333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16
4,1,2024-02-01,44.2,0.721,1,4,1.0,0.0,0.0,0.0,3.0,44.333333,0.333333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16


Panel shape after feature merges: (534, 18)


### Create the future target carefully

This is one of the most important blocks in the notebook.

The target is based on **next-month** education outcomes, but the predictors come from the **current month**.  
That separation is essential to reduce leakage and keep the model aligned with a real prediction task.

In [14]:

# Create next-month target and future education outcomes.
# The model predicts this target using current-month information only.

panel = resident_month_panel.copy()

# Standardize the education and attendance column names before creating the target.
if "education_progress" not in panel.columns:
    if "avg_education_progress" in panel.columns:
        panel = panel.rename(columns={"avg_education_progress": "education_progress"})
    elif "education_progress_score" in panel.columns:
        panel = panel.rename(columns={"education_progress_score": "education_progress"})

if "attendance_rate" not in panel.columns:
    if "avg_attendance_rate" in panel.columns:
        panel = panel.rename(columns={"avg_attendance_rate": "attendance_rate"})
    elif "attendance" in panel.columns:
        panel = panel.rename(columns={"attendance": "attendance_rate"})

panel = panel.sort_values(["resident_id", "metric_month"]).reset_index(drop=True)

if "education_progress" not in panel.columns or "attendance_rate" not in panel.columns:
    raise KeyError(
        f"Missing required columns for target creation. Available columns: {panel.columns.tolist()}"
    )

panel["next_month_progress"] = panel.groupby("resident_id")["education_progress"].shift(-1)
panel["next_month_attendance_rate"] = panel.groupby("resident_id")["attendance_rate"].shift(-1)
panel["next_metric_month"] = panel.groupby("resident_id")["metric_month"].shift(-1)

# Only keep labels where the next record is exactly the next calendar month.
expected_next_month = panel["metric_month"] + pd.offsets.MonthBegin(1)
panel["has_true_next_month"] = panel["next_metric_month"].eq(expected_next_month)

panel["milestone_met_next_month"] = np.where(
    panel["has_true_next_month"]
    & panel["next_month_progress"].ge(70)
    & panel["next_month_attendance_rate"].ge(0.80),
    1,
    np.where(panel["has_true_next_month"], 0, np.nan)
)

display(
    panel[
        [
            "resident_id",
            "metric_month",
            "education_progress",
            "attendance_rate",
            "next_month_progress",
            "next_month_attendance_rate",
            "milestone_met_next_month",
        ]
    ].head(12)
)

,resident_id,metric_month,education_progress,attendance_rate,next_month_progress,next_month_attendance_rate,milestone_met_next_month
0,1,2023-10-01,37.7,0.966,33.0,0.693,0.0
1,1,2023-11-01,33.0,0.693,54.0,0.744,0.0
2,1,2023-12-01,54.0,0.744,51.2,0.681,0.0
3,1,2024-01-01,51.2,0.681,44.2,0.721,0.0
4,1,2024-02-01,44.2,0.721,52.8,0.493,0.0
5,1,2024-03-01,52.8,0.493,NaN,NaN,NaN
6,2,2023-03-01,66.8,0.626,73.7,0.758,0.0
7,2,2023-04-01,73.7,0.758,83.2,0.923,1.0
8,2,2023-05-01,83.2,0.923,72.9,0.872,1.0
9,2,2023-06-01,72.9,0.872,66.0,0.820,0.0


### Lagged, rolling, and calendar features

This block creates history-based features from prior resident-month records.
These features are useful because education momentum is often path-dependent rather than purely cross-sectional.

In [15]:

# Add lagged, rolling, and calendar features.
panel = panel.sort_values(["resident_id", "metric_month"]).reset_index(drop=True)

lag_cols = [
    col for col in [
        "education_progress",
        "attendance_rate",
        "education_record_count",
        "incident_count",
        "high_severity_incidents",
        "health_score",
        "sleep_quality_score",
        "energy_level_score",
        "process_session_count",
        "avg_session_duration",
        "plans_created",
        "achieved_plans",
        "education_plans_created",
    ] if col in panel.columns
]

for col in lag_cols:
    panel[f"{col}_lag1"] = panel.groupby("resident_id")[col].shift(1)
    panel[f"{col}_lag2"] = panel.groupby("resident_id")[col].shift(2)

rolling_candidates = [col for col in ["education_progress", "attendance_rate", "incident_count"] if col in panel.columns]
for col in rolling_candidates:
    panel[f"{col}_roll3_mean"] = (
        panel.groupby("resident_id")[col]
        .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
    )
    panel[f"{col}_trend_vs_lag1"] = panel[col] - panel.groupby("resident_id")[col].shift(1)

panel["month_num"] = panel["metric_month"].dt.month
panel["year_num"] = panel["metric_month"].dt.year
panel["resident_month_index"] = panel.groupby("resident_id").cumcount() + 1

if "open_date" in panel.columns:
    panel["safehouse_age_days"] = (panel["metric_month"] - panel["open_date"]).dt.days

# Add a simple current-month milestone flag for context.
panel["milestone_met_current_month"] = np.where(
    panel["education_progress"].ge(70) & panel["attendance_rate"].ge(0.80),
    1,
    0
)

display(panel.head())

,resident_id,metric_month,education_progress,attendance_rate,education_record_count,safehouse_id,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents,process_session_count,avg_session_duration,group_session_share,plans_created,achieved_plans,name,region,open_date,next_month_progress,next_month_attendance_rate,next_metric_month,has_true_next_month,milestone_met_next_month,education_progress_lag1,education_progress_lag2,attendance_rate_lag1,attendance_rate_lag2,education_record_count_lag1,education_record_count_lag2,incident_count_lag1,incident_count_lag2,high_severity_incidents_lag1,high_severity_incidents_lag2,process_session_count_lag1,process_session_count_lag2,avg_session_duration_lag1,avg_session_duration_lag2,plans_created_lag1,plans_created_lag2,achieved_plans_lag1,achieved_plans_lag2,education_progress_roll3_mean,education_progress_trend_vs_lag1,attendance_rate_roll3_mean,attendance_rate_trend_vs_lag1,incident_count_roll3_mean,incident_count_trend_vs_lag1,month_num,year_num,resident_month_index,safehouse_age_days,milestone_met_current_month
0,1,2023-10-01,37.7,0.966,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16,33.0,0.693,2023-11-01,True,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,2023,1,503,0
1,1,2023-11-01,33.0,0.693,1,4,NaN,NaN,NaN,NaN,3.0,74.000000,0.333333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16,54.0,0.744,2023-12-01,True,0.0,37.7,NaN,0.966,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,0.0,NaN,37.700000,-4.7,0.9660,-0.273,NaN,NaN,11,2023,2,534,0
2,1,2023-12-01,54.0,0.744,1,4,NaN,NaN,NaN,NaN,6.0,75.333333,0.833333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16,51.2,0.681,2024-01-01,True,0.0,33.0,37.7,0.693,0.966,1.0,1.0,NaN,NaN,NaN,NaN,3.0,NaN,74.000000,NaN,NaN,3.0,NaN,0.0,35.350000,21.0,0.8295,0.051,NaN,NaN,12,2023,3,564,0
3,1,2024-01-01,51.2,0.681,1,4,NaN,NaN,NaN,NaN,3.0,60.666667,0.333333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16,44.2,0.721,2024-02-01,True,0.0,54.0,33.0,0.744,0.693,1.0,1.0,NaN,NaN,NaN,NaN,6.0,3.0,75.333333,74.000000,NaN,NaN,NaN,NaN,41.566667,-2.8,0.8010,-0.063,NaN,NaN,1,2024,4,595,0
4,1,2024-02-01,44.2,0.721,1,4,1.0,0.0,0.0,0.0,3.0,44.333333,0.333333,NaN,NaN,Lighthouse Safehouse 4,Visayas,2022-05-16,52.8,0.493,2024-03-01,True,0.0,51.2,54.0,0.681,0.744,1.0,1.0,NaN,NaN,NaN,NaN,3.0,6.0,60.666667,75.333333,NaN,NaN,NaN,NaN,46.066667,-7.0,0.7060,0.040,NaN,NaN,2,2024,5,626,0


### Final panel cleanup before modeling

This block:
- cleans numeric fields
- fills selected missing values
- removes rows that cannot support the prediction task
- prepares the final modeling table

In [16]:

# Clean obvious numeric fields and prepare the final modeling dataset.
panel_model = panel.copy()

# Convert likely numeric columns.
for col in panel_model.columns:
    if col in ["resident_id", "safehouse_id", "name", "region"]:
        continue
    if panel_model[col].dtype == "object":
        continue
    if str(panel_model[col].dtype).startswith("datetime"):
        continue
    panel_model[col] = pd.to_numeric(panel_model[col], errors="ignore")

# Fill missing count-like features with zero before modeling.
zero_fill_keywords = [
    "count", "plans", "incidents", "record_count", "session_count", "flag", "share"
]
zero_fill_cols = [
    col for col in panel_model.columns
    if any(keyword in col for keyword in zero_fill_keywords)
]
for col in zero_fill_cols:
    if col in panel_model.columns:
        panel_model[col] = panel_model[col].fillna(0)

# Keep only rows with a valid next-month target.
panel_model = panel_model.dropna(subset=["milestone_met_next_month"]).copy()
panel_model["milestone_met_next_month"] = panel_model["milestone_met_next_month"].astype(int)

# Save a copy of the modeling panel for review.
panel_output_path = OUTPUT_DIR / "education_outcome_prediction_panel.csv"
panel_model.to_csv(panel_output_path, index=False)

print("Modeling panel shape:", panel_model.shape)
print("Panel saved to:", panel_output_path)
display(panel_model.head())

Modeling panel shape: (474, 52)
Panel saved to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\education_outcome_prediction_panel.csv


,resident_id,metric_month,education_progress,attendance_rate,education_record_count,safehouse_id,incident_count,high_severity_incidents,medium_severity_incidents,follow_up_incidents,process_session_count,avg_session_duration,group_session_share,plans_created,achieved_plans,name,region,open_date,next_month_progress,next_month_attendance_rate,next_metric_month,has_true_next_month,milestone_met_next_month,education_progress_lag1,education_progress_lag2,attendance_rate_lag1,attendance_rate_lag2,education_record_count_lag1,education_record_count_lag2,incident_count_lag1,incident_count_lag2,high_severity_incidents_lag1,high_severity_incidents_lag2,process_session_count_lag1,process_session_count_lag2,avg_session_duration_lag1,avg_session_duration_lag2,plans_created_lag1,plans_created_lag2,achieved_plans_lag1,achieved_plans_lag2,education_progress_roll3_mean,education_progress_trend_vs_lag1,attendance_rate_roll3_mean,attendance_rate_trend_vs_lag1,incident_count_roll3_mean,incident_count_trend_vs_lag1,month_num,year_num,resident_month_index,safehouse_age_days,milestone_met_current_month
0,1,2023-10-01,37.7,0.966,1,4,0.0,0.0,0.0,0.0,0.0,NaN,0.000000,3.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16,33.0,0.693,2023-11-01,True,0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,10,2023,1,503,0
1,1,2023-11-01,33.0,0.693,1,4,0.0,0.0,0.0,0.0,3.0,74.000000,0.333333,0.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16,54.0,0.744,2023-12-01,True,0,37.7,NaN,0.966,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,3.0,0.0,0.0,0.0,37.700000,-4.7,0.9660,-0.273,0.0,0.0,11,2023,2,534,0
2,1,2023-12-01,54.0,0.744,1,4,0.0,0.0,0.0,0.0,6.0,75.333333,0.833333,0.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16,51.2,0.681,2024-01-01,True,0,33.0,37.7,0.693,0.966,1.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0,74.000000,NaN,0.0,3.0,0.0,0.0,35.350000,21.0,0.8295,0.051,0.0,0.0,12,2023,3,564,0
3,1,2024-01-01,51.2,0.681,1,4,0.0,0.0,0.0,0.0,3.0,60.666667,0.333333,0.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16,44.2,0.721,2024-02-01,True,0,54.0,33.0,0.744,0.693,1.0,1.0,0.0,0.0,0.0,0.0,6.0,3.0,75.333333,74.000000,0.0,0.0,0.0,0.0,41.566667,-2.8,0.8010,-0.063,0.0,0.0,1,2024,4,595,0
4,1,2024-02-01,44.2,0.721,1,4,1.0,0.0,0.0,0.0,3.0,44.333333,0.333333,0.0,0.0,Lighthouse Safehouse 4,Visayas,2022-05-16,52.8,0.493,2024-03-01,True,0,51.2,54.0,0.681,0.744,1.0,1.0,0.0,0.0,0.0,0.0,3.0,6.0,60.666667,75.333333,0.0,0.0,0.0,0.0,46.066667,-7.0,0.7060,0.040,0.0,0.0,2,2024,5,626,0


### Exploratory checks on class balance and time coverage

Before modeling, it is important to confirm:
- whether the target is highly imbalanced
- how many rows exist across months
- whether the prediction task has enough time coverage to support a time-aware split

In [17]:

# Review class balance and time coverage.
class_balance = (
    panel_model["milestone_met_next_month"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="rows")
)
class_balance["share"] = class_balance["rows"] / class_balance["rows"].sum()

display(class_balance)

month_summary = (
    panel_model.groupby("metric_month", as_index=False)
    .agg(
        rows=("resident_id", "size"),
        positive_rate=("milestone_met_next_month", "mean"),
    )
    .sort_values("metric_month")
)

display(month_summary.tail(12))

,target,rows,share
0,0,331,0.698312
1,1,143,0.301688


,metric_month,rows,positive_rate
25,2025-02-01,13,0.307692
26,2025-03-01,16,0.250000
27,2025-04-01,18,0.388889
28,2025-05-01,15,0.266667
29,2025-06-01,14,0.214286
30,2025-07-01,11,0.181818
31,2025-08-01,9,0.111111
32,2025-09-01,7,0.571429
33,2025-10-01,5,0.200000
34,2025-11-01,3,0.333333


## 3. Modeling & Feature Selection

### Time-aware split
This notebook uses a **time-aware train/test split** rather than a random split.
That better matches the real deployment setting because the model should predict future resident outcomes using past data.

In [18]:

# Create train and test sets using a time-aware split.
# The latest 20 percent of months become the test period.

unique_months = sorted(panel_model["metric_month"].dropna().unique())
test_month_count = max(1, math.ceil(len(unique_months) * 0.20))

test_months = unique_months[-test_month_count:]
train_months = unique_months[:-test_month_count]

train_df = panel_model[panel_model["metric_month"].isin(train_months)].copy()
test_df = panel_model[panel_model["metric_month"].isin(test_months)].copy()

# Fall back to a simple split if there are too few months.
if train_df.empty or test_df.empty:
    cutoff = int(len(panel_model) * 0.80)
    panel_model = panel_model.sort_values(["metric_month", "resident_id"]).reset_index(drop=True)
    train_df = panel_model.iloc[:cutoff].copy()
    test_df = panel_model.iloc[cutoff:].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train months:", train_df["metric_month"].min(), "to", train_df["metric_month"].max())
print("Test months:", test_df["metric_month"].min(), "to", test_df["metric_month"].max())

Train shape: (423, 52)
Test shape: (51, 52)
Train months: 2023-01-01 00:00:00 to 2025-05-01 00:00:00
Test months: 2025-06-01 00:00:00 to 2026-01-01 00:00:00


### Leakage control and feature definition

This block explicitly removes:
- future outcome columns
- target-derived fields
- columns that would leak next-month information

It also separates the final feature matrix from the target.

In [19]:

# Define the feature matrix and drop columns that would leak future information or cause model issues.

target_col = "milestone_met_next_month"

leakage_cols = [
    "next_month_progress",
    "next_month_attendance_rate",
    "next_metric_month",
    "has_true_next_month",
    target_col,
]

identifier_cols = [
    "resident_id",
]

# Keep safehouse_id as a categorical feature if present.
# Keep region and safehouse name if available because they can add context.
drop_cols = [col for col in leakage_cols + identifier_cols if col in train_df.columns]

X_train = train_df.drop(columns=drop_cols, errors="ignore").copy()
X_test = test_df.drop(columns=drop_cols, errors="ignore").copy()

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

# Drop raw datetime columns from the feature matrix.
datetime_cols = X_train.select_dtypes(include=["datetime64[ns]", "datetime64", "datetimetz"]).columns.tolist()
X_train = X_train.drop(columns=datetime_cols, errors="ignore")
X_test = X_test.drop(columns=datetime_cols, errors="ignore")

# Align columns in case one side lost a column after cleaning.
common_cols = sorted(set(X_train.columns).intersection(set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

# Use resident_id groups in cross-validation so the same resident does not appear in both train and validation folds.
groups_train = train_df["resident_id"].copy()

print("Dropped datetime columns:", datetime_cols)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
display(X_train.head())

Dropped datetime columns: ['metric_month', 'open_date']
X_train shape: (423, 44)
X_test shape: (51, 44)


,achieved_plans,achieved_plans_lag1,achieved_plans_lag2,attendance_rate,attendance_rate_lag1,attendance_rate_lag2,attendance_rate_roll3_mean,attendance_rate_trend_vs_lag1,avg_session_duration,avg_session_duration_lag1,avg_session_duration_lag2,education_progress,education_progress_lag1,education_progress_lag2,education_progress_roll3_mean,education_progress_trend_vs_lag1,education_record_count,education_record_count_lag1,education_record_count_lag2,follow_up_incidents,group_session_share,high_severity_incidents,high_severity_incidents_lag1,high_severity_incidents_lag2,incident_count,incident_count_lag1,incident_count_lag2,incident_count_roll3_mean,incident_count_trend_vs_lag1,medium_severity_incidents,milestone_met_current_month,month_num,name,plans_created,plans_created_lag1,plans_created_lag2,process_session_count,process_session_count_lag1,process_session_count_lag2,region,resident_month_index,safehouse_age_days,safehouse_id,year_num
0,0.0,0.0,0.0,0.966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.7,NaN,NaN,NaN,NaN,1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10,Lighthouse Safehouse 4,3.0,0.0,0.0,0.0,0.0,0.0,Visayas,1,503,4,2023
1,0.0,0.0,0.0,0.693,0.966,NaN,0.9660,-0.273,74.000000,NaN,NaN,33.0,37.7,NaN,37.700000,-4.7,1,1.0,0.0,0.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,11,Lighthouse Safehouse 4,0.0,3.0,0.0,3.0,0.0,0.0,Visayas,2,534,4,2023
2,0.0,0.0,0.0,0.744,0.693,0.966,0.8295,0.051,75.333333,74.000000,NaN,54.0,33.0,37.7,35.350000,21.0,1,1.0,1.0,0.0,0.833333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,12,Lighthouse Safehouse 4,0.0,0.0,3.0,6.0,3.0,0.0,Visayas,3,564,4,2023
3,0.0,0.0,0.0,0.681,0.744,0.693,0.8010,-0.063,60.666667,75.333333,74.000000,51.2,54.0,33.0,41.566667,-2.8,1,1.0,1.0,0.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,Lighthouse Safehouse 4,0.0,0.0,0.0,3.0,6.0,3.0,Visayas,4,595,4,2024
4,0.0,0.0,0.0,0.721,0.681,0.744,0.7060,0.040,44.333333,60.666667,75.333333,44.2,51.2,54.0,46.066667,-7.0,1,1.0,1.0,0.0,0.333333,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,2,Lighthouse Safehouse 4,0.0,0.0,0.0,3.0,3.0,6.0,Visayas,5,626,4,2024


### Feature typing

This block identifies numeric and categorical features for the preprocessing pipeline.
It also treats identifier-like safehouse fields carefully so they are encoded appropriately rather than scaled as continuous quantities.

In [20]:

# Recompute numeric and categorical feature lists after feature cleanup.
numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Include safehouse_id as categorical if it remained numeric but represents a label-like ID.
if "safehouse_id" in X_train.columns and "safehouse_id" in numeric_features:
    numeric_features.remove("safehouse_id")
    categorical_features.append("safehouse_id")

categorical_features = sorted(set(categorical_features))

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))
print("Categorical features:", categorical_features)

Numeric feature count: 41
Categorical feature count: 3
Categorical features: ['name', 'region', 'safehouse_id']


### Preprocessing pipeline

This block builds the reusable preprocessing pipeline:
- numeric imputation and scaling
- categorical imputation and one-hot encoding

This is the main place where dummy handling is implemented in a repeatable way.

In [21]:

# Build preprocessing pipelines.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

### Candidate models

This block defines the candidate classifiers used for comparison.
The notebook uses multiple models so the final choice is justified rather than assumed.

In [22]:

# Define candidate models.
logit_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED, class_weight="balanced")),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=4,
            random_state=SEED,
            class_weight="balanced_subsample",
            n_jobs=-1,
        )),
    ]
)

MODELS = {
    "logistic_regression": logit_pipeline,
    "random_forest": rf_pipeline,
}

### Grouped cross-validation

This section evaluates candidate models using grouped cross-validation by resident.
That helps reduce resident-level leakage across folds and gives a more trustworthy selection process.

In [23]:

# Grouped cross-validation on the training set.
# This protects against resident-level leakage across folds.

def evaluate_group_cv(model, X, y, groups, n_splits=5) -> list[dict]:
    """Return fold-level metrics using GroupKFold."""
    n_unique_groups = pd.Series(groups).nunique()
    n_splits = min(n_splits, int(n_unique_groups))
    if n_splits < 2:
        raise ValueError("Grouped cross-validation requires at least two unique resident_id groups.")

    cv = GroupKFold(n_splits=n_splits)
    fold_results = []

    for fold_num, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups=groups), start=1):
        X_tr = X.iloc[train_idx].copy()
        X_val = X.iloc[valid_idx].copy()
        y_tr = y.iloc[train_idx].copy()
        y_val = y.iloc[valid_idx].copy()

        fitted = clone(model)
        fitted.fit(X_tr, y_tr)

        val_pred = fitted.predict(X_val)
        val_proba = fitted.predict_proba(X_val)[:, 1]

        fold_results.append(
            {
                "fold": fold_num,
                "accuracy": accuracy_score(y_val, val_pred),
                "precision": precision_score(y_val, val_pred, zero_division=0),
                "recall": recall_score(y_val, val_pred, zero_division=0),
                "f1": f1_score(y_val, val_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_val, val_proba) if y_val.nunique() > 1 else np.nan,
                "avg_precision": average_precision_score(y_val, val_proba),
            }
        )

    return fold_results


cv_rows = []
for model_name, model in MODELS.items():
    fold_rows = evaluate_group_cv(model, X_train, y_train, groups_train, n_splits=5)
    for row in fold_rows:
        row["model_name"] = model_name
        cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows)
display(cv_results)

cv_summary = (
    cv_results
    .groupby("model_name", as_index=False)
    .agg({
        "accuracy": ["mean", "std"],
        "precision": ["mean", "std"],
        "recall": ["mean", "std"],
        "f1": ["mean", "std"],
        "roc_auc": ["mean", "std"],
        "avg_precision": ["mean", "std"],
    })
)

cv_summary.columns = [
    "_".join([piece for piece in col if piece]).strip("_")
    for col in cv_summary.columns.to_flat_index()
]

display(cv_summary.sort_values("f1_mean", ascending=False))

,fold,accuracy,precision,recall,f1,roc_auc,avg_precision,model_name
0,1,0.647059,0.307692,0.400000,0.347826,0.693846,0.345996,logistic_regression
1,2,0.588235,0.315789,0.571429,0.406780,0.662202,0.382007,logistic_regression
2,3,0.654762,0.542857,0.593750,0.567164,0.736178,0.637110,logistic_regression
3,4,0.588235,0.413043,0.703704,0.520548,0.648787,0.449159,logistic_regression
4,5,0.630952,0.486486,0.600000,0.537313,0.652469,0.494074,logistic_regression
5,1,0.670588,0.333333,0.400000,0.363636,0.736154,0.413580,random_forest
6,2,0.635294,0.343750,0.523810,0.415094,0.729167,0.429752,random_forest
7,3,0.654762,0.545455,0.562500,0.553846,0.716346,0.576850,random_forest
8,4,0.576471,0.404255,0.703704,0.513514,0.664112,0.434220,random_forest
9,5,0.654762,0.514286,0.600000,0.553846,0.668519,0.481983,random_forest


,model_name,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,avg_precision_mean,avg_precision_std
1,random_forest,0.638375,0.036798,0.428216,0.097294,0.558003,0.110845,0.479987,0.086289,0.702860,0.034143,0.467277,0.066345
0,logistic_regression,0.621849,0.031865,0.413174,0.103444,0.573776,0.109740,0.475926,0.093908,0.678696,0.036702,0.461669,0.113702


### Select the final model

This block selects the best-performing model based on cross-validation results and then fits it on the full training set.

In [24]:

# Select the best model using mean F1, then fit on the full training data.
best_model_name = cv_summary.sort_values(["f1_mean", "roc_auc_mean"], ascending=False).iloc[0]["model_name"]
best_model = clone(MODELS[best_model_name])
best_model.fit(X_train, y_train)

print("Selected model:", best_model_name)

Selected model: random_forest


## 4. Evaluation & Interpretation

This section evaluates the selected model on the held-out test period.

### Error meaning in this context
- **False positive**: staff may spend extra time supporting a resident who would have met the milestone anyway
- **False negative**: staff may miss a resident who needed earlier support

In this use case, false negatives are usually more costly because missing at-risk residents weakens intervention value.

In [25]:

# Evaluate the selected model on the held-out test set.
test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame(
    [
        {
            "model_name": best_model_name,
            "accuracy": accuracy_score(y_test, test_pred),
            "precision": precision_score(y_test, test_pred, zero_division=0),
            "recall": recall_score(y_test, test_pred, zero_division=0),
            "f1": f1_score(y_test, test_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_proba) if y_test.nunique() > 1 else np.nan,
            "avg_precision": average_precision_score(y_test, test_proba),
        }
    ]
)

display(test_metrics)
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))
print()
print("Classification report:")
print(classification_report(y_test, test_pred, zero_division=0))

,model_name,accuracy,precision,recall,f1,roc_auc,avg_precision
0,random_forest,0.54902,0.34375,0.846154,0.488889,0.67004,0.356715


Confusion matrix:
[[17 21]
 [ 2 11]]

Classification report:
              precision    recall  f1-score   support

           0       0.89      0.45      0.60        38
           1       0.34      0.85      0.49        13

    accuracy                           0.55        51
   macro avg       0.62      0.65      0.54        51
weighted avg       0.75      0.55      0.57        51



### Save core evaluation outputs

These outputs make the notebook easier to connect to the web app, reports, and final presentation materials.

In [26]:

# Save evaluation outputs.
cv_output_path = OUTPUT_DIR / "education_outcome_prediction_cv_results.csv"
cv_summary_path = OUTPUT_DIR / "education_outcome_prediction_cv_summary.csv"
test_metrics_path = OUTPUT_DIR / "education_outcome_prediction_test_metrics.csv"

cv_results.to_csv(cv_output_path, index=False)
cv_summary.to_csv(cv_summary_path, index=False)
test_metrics.to_csv(test_metrics_path, index=False)

print("Saved CV results to:", cv_output_path)
print("Saved CV summary to:", cv_summary_path)
print("Saved test metrics to:", test_metrics_path)

Saved CV results to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\education_outcome_prediction_cv_results.csv
Saved CV summary to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\education_outcome_prediction_cv_summary.csv
Saved test metrics to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\education_outcome_prediction_test_metrics.csv


## 5. Causal and Relationship Analysis

This section does **not** claim causal proof.
Instead, it examines which features appear most important to the selected predictive model and discusses whether the discovered relationships make practical sense.

In [27]:

# Estimate feature importance with permutation importance on the test set.
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    scoring="f1",
    n_jobs=-1,
)

importance_df = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values("importance_mean", ascending=False)

display(importance_df.head(25))

importance_output_path = OUTPUT_DIR / "education_outcome_prediction_feature_importance.csv"
importance_df.to_csv(importance_output_path, index=False)
print("Saved feature importance to:", importance_output_path)

,feature,importance_mean,importance_std
11,education_progress,0.075427,0.036000
32,name,0.052149,0.029335
42,safehouse_id,0.037714,0.031425
30,milestone_met_current_month,0.036135,0.019627
4,attendance_rate_lag1,0.032530,0.028488
8,avg_session_duration,0.018492,0.015584
5,attendance_rate_lag2,0.016839,0.031573
31,month_num,0.014109,0.017228
41,safehouse_age_days,0.013996,0.021572
36,process_session_count,0.013458,0.033671


Saved feature importance to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\education_outcome_prediction_feature_importance.csv


## 6. Deployment Notes

This block scores the latest eligible resident-month records so the web application can surface actionable outputs.

### Suggested page integration
This pipeline fits best in:
- **Caseload Inventory**
- **Admin Dashboard**
- **Reports & Analytics**

Typical UI uses:
- risk flags
- milestone probabilities
- prioritized support queues

In [28]:

# Score the latest eligible resident-month records for action planning.
# This output is intended for dashboard or admin review.

latest_month = panel_model["metric_month"].max()
latest_candidates = panel_model[panel_model["metric_month"] == latest_month].copy()

latest_drop_cols = [col for col in drop_cols if col in latest_candidates.columns]
X_latest = latest_candidates.drop(columns=latest_drop_cols, errors="ignore").copy()

latest_datetime_cols = X_latest.select_dtypes(include=["datetime64[ns]", "datetime64", "datetimetz"]).columns.tolist()
X_latest = X_latest.drop(columns=latest_datetime_cols, errors="ignore")

for col in common_cols:
    if col not in X_latest.columns:
        X_latest[col] = np.nan

X_latest = X_latest[common_cols].copy()

latest_candidates["milestone_probability_next_month"] = best_model.predict_proba(X_latest)[:, 1]
latest_candidates["predicted_milestone_next_month"] = best_model.predict(X_latest)

score_cols = [
    col for col in [
        "resident_id",
        "safehouse_id",
        "name",
        "region",
        "metric_month",
        "education_progress",
        "attendance_rate",
        "incident_count",
        "health_score",
        "process_session_count",
        "plans_created",
        "milestone_probability_next_month",
        "predicted_milestone_next_month",
    ] if col in latest_candidates.columns
]

scored_latest = latest_candidates[score_cols].sort_values(
    ["milestone_probability_next_month", "resident_id"],
    ascending=[False, True]
).reset_index(drop=True)

display(scored_latest.head(25))

scores_output_path = OUTPUT_DIR / "education_outcome_prediction_scores.csv"
scored_latest.to_csv(scores_output_path, index=False)

print("Latest scored output saved to:", scores_output_path)

,resident_id,safehouse_id,name,region,metric_month,education_progress,attendance_rate,incident_count,process_session_count,plans_created,milestone_probability_next_month,predicted_milestone_next_month
0,40,1,Lighthouse Safehouse 1,Luzon,2026-01-01,100.0,0.886,0.0,4.0,0.0,0.668564,1


Latest scored output saved to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\education_outcome_prediction_scores.csv


## Final integration notes

The saved score file is the most useful artifact for website integration.
The panel and summary outputs are mainly helpful for debugging, documentation, and presentation support.

## Notes for project integration

- The scoring file in `generated_outputs/education_outcome_prediction_scores.csv` is the cleanest output to connect to an admin dashboard.
- The panel file in `generated_outputs/education_outcome_prediction_panel.csv` is useful for debugging and documentation.
- If your team uses slightly different CSV names or column names, update the alias maps near the top of the notebook rather than changing later modeling cells.
- If class balance is very uneven, threshold tuning can be added after the baseline pipeline is validated.

In [ ]:
# Mirror outputs back to the local repo fallback after writing to Azure ML output storage when available.
import shutil
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "generated_outputs"
if OUTPUT_DIR.resolve() != LOCAL_OUTPUT_DIR.resolve():
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for artifact in OUTPUT_DIR.glob('*'):
        if artifact.is_file():
            shutil.copy2(artifact, LOCAL_OUTPUT_DIR / artifact.name)
    print(f'Mirrored outputs to local fallback: {LOCAL_OUTPUT_DIR.resolve()}')
